In [ ]:

import pandas as pd
import numpy as np
import tensorflow as tf
import keras
from keras import layers, Model, callbacks
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.model_selection import train_test_split
from collections import Counter
import gc

# FIX KERAS 3 : mode eager obligatoire
tf.config.run_functions_eagerly(True)

# Seed pour la reproductibilité
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================================================
# 1. PRÉPARATION, NETTOYAGE ET MAPPING
# =========================================================
gc.collect()

path = "C:/Users/Théo Lassale/Desktop/Perso/Stage/Stage_Malmö/IMU_LM_Data/data/merged_dataset/unified_dataset.parquet"

# Chargement uniquement des colonnes nécessaires
df = pd.read_parquet(path, columns=["acc_x", "acc_y", "acc_z", "global_activity_id",
                                     "dataset", "subject_id", "session_id"])

# Conversion float32 pour économiser la RAM
for col in ["acc_x", "acc_y", "acc_z"]:
    df[col] = df[col].astype(np.float32)

gc.collect()

# Suppression des labels aberrants et Mapping consécutif (0, 1, 2...)
df = df[df["global_activity_id"] < 100].copy()
unique_labels = sorted(df["global_activity_id"].unique())
label_mapping = {old: new for new, old in enumerate(unique_labels)}
df["global_activity_id"] = df["global_activity_id"].map(label_mapping)

print(f"Nombre de classes détectées : {len(unique_labels)}")

features = ["acc_x", "acc_y", "acc_z"]

# À adapter selon la fréquence native du capteur :
# - Si 50Hz natif → garder cette ligne (passe à 25Hz)
# - Si déjà 25Hz  → commenter cette ligne
df = df.iloc[::2].copy()

df[features] = df[features].interpolate().bfill().ffill()

WINDOW_SIZE, STRIDE = 50, 25
X, labels = [], []

for _, group in df.groupby(["dataset", "subject_id", "session_id"]):
    sig = group[features].values
    acts = group["global_activity_id"].values
    for i in range(0, len(sig) - WINDOW_SIZE, STRIDE):
        X.append(sig[i:i + WINDOW_SIZE])
        most_common = Counter(acts[i:i + WINDOW_SIZE]).most_common(1)[0][0]
        labels.append(most_common)

X = np.array(X, dtype=np.float32)
labels = np.array(labels)

# Libère la RAM du dataframe
del df
gc.collect()

X_temp, X_test, y_temp, y_test = train_test_split(X, labels, test_size=0.10, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2222, random_state=SEED)

del X_temp, y_temp
gc.collect()

mean, std = np.mean(X_train, axis=(0, 1)), np.std(X_train, axis=(0, 1))
X_train = (X_train - mean) / (std + 1e-7)
X_val   = (X_val   - mean) / (std + 1e-7)
X_test  = (X_test  - mean) / (std + 1e-7)

timesteps, n_features, latent_dim = 50, 3, 32

# =========================================================
# 2. VAE GÉNÉRATIF — Compatible Keras 3
# =========================================================
def get_encoder():
    inputs = layers.Input(shape=(timesteps, n_features))
    x = layers.Conv1D(32, 5, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    return Model(inputs, [layers.Dense(latent_dim)(x), layers.Dense(latent_dim)(x)])

def get_decoder():
    inputs = layers.Input(shape=(latent_dim,))
    x = layers.Dense(13 * 128, activation='relu')(inputs)
    x = layers.Reshape((13, 128))(x)
    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.UpSampling1D(2)(x)
    x = layers.Cropping1D(cropping=(1, 1))(x)
    x = layers.Conv1D(32, 5, activation='relu', padding='same')(x)
    outputs = layers.Conv1D(n_features, 5, activation='linear', padding='same')(x)
    return Model(inputs, outputs)

class VAE(Model):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc, self.dec = enc, dec
        # FIX KERAS 3 : déclarer les métriques explicitement
        self.loss_tracker  = keras.metrics.Mean(name="loss")
        self.recon_tracker = keras.metrics.Mean(name="recon")

    @property
    def metrics(self):
        return [self.loss_tracker, self.recon_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            zm, zv = self.enc(data, training=True)
            z = zm + tf.exp(0.5 * zv) * tf.random.normal(tf.shape(zm))
            recon = self.dec(z, training=True)
            r_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.mse(data, recon), axis=-1))
            kl_loss = -0.5 * tf.reduce_mean(1 + zv - tf.square(zm) - tf.exp(zv))
            total = r_loss + 0.01 * kl_loss
        self.optimizer.apply_gradients(
            zip(tape.gradient(total, self.trainable_weights), self.trainable_weights)
        )
        # FIX KERAS 3 : update_state + result()
        self.loss_tracker.update_state(total)
        self.recon_tracker.update_state(r_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        zm, zv = self.enc(data, training=False)
        z = zm + tf.exp(0.5 * zv) * tf.random.normal(tf.shape(zm))
        recon = self.dec(z, training=False)
        r_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.mse(data, recon), axis=-1))
        kl_loss = -0.5 * tf.reduce_mean(1 + zv - tf.square(zm) - tf.exp(zv))
        total = r_loss + 0.01 * kl_loss
        self.loss_tracker.update_state(total)
        self.recon_tracker.update_state(r_loss)
        return {m.name: m.result() for m in self.metrics}

vae_enc, vae_dec = get_encoder(), get_decoder()
vae_model = VAE(vae_enc, vae_dec)
vae_model.compile(optimizer='adam')

# tf.data.Dataset pour éviter les problèmes de tuple
train_ds = tf.data.Dataset.from_tensor_slices(X_train).batch(256).prefetch(tf.data.AUTOTUNE)
val_ds   = tf.data.Dataset.from_tensor_slices(X_val).batch(256).prefetch(tf.data.AUTOTUNE)

# Callback dédié au VAE
early_stop_vae = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("\n--- Entraînement VAE ---")
vae_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop_vae],
    verbose=1
)

# =========================================================
# 3. LES 3 CLASSIFIERS (JUGES)
# =========================================================
def build_judges(n_classes):
    i = layers.Input(shape=(50, 3))

    # Juge 1 : DeepConvLSTM
    x1 = layers.Conv1D(64, 5, activation='relu', padding='same')(i)
    x1 = layers.Conv1D(64, 5, activation='relu', padding='same')(x1)
    x1 = layers.LSTM(128, return_sequences=True)(x1)
    x1 = layers.LSTM(128)(x1)
    x1 = layers.Dense(128, activation='relu')(x1)
    x1 = layers.Dropout(0.3)(x1)
    m1 = Model(i, layers.Dense(n_classes, activation='softmax')(x1), name="DeepConvLSTM")

    # Juge 2 : CNN Simple
    x2 = layers.Conv1D(64, 3, activation='relu', padding='same')(i)
    x2 = layers.Conv1D(64, 3, activation='relu', padding='same')(x2)
    x2 = layers.GlobalAveragePooling1D()(x2)
    x2 = layers.Dense(128, activation='relu')(x2)
    x2 = layers.Dropout(0.3)(x2)
    m2 = Model(i, layers.Dense(n_classes, activation='softmax')(x2), name="CNN_Simple")

    # Juge 3 : MLP Simple
    x3 = layers.Flatten()(i)
    x3 = layers.Dense(256, activation='relu')(x3)
    x3 = layers.Dropout(0.3)(x3)
    x3 = layers.Dense(128, activation='relu')(x3)
    x3 = layers.Dropout(0.3)(x3)
    m3 = Model(i, layers.Dense(n_classes, activation='softmax')(x3), name="MLP_Simple")

    return [m1, m2, m3]

judges = build_judges(len(unique_labels))

for m in judges:
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f"\nEntraînement du juge {m.name}...")

    # Nouveau callback indépendant pour chaque juge
    early_stop_judge = callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        mode='max',
        restore_best_weights=True
    )

    m.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=256,
        callbacks=[early_stop_judge],
        verbose=1
    )

# =========================================================
# 4. ANALYSE GAIT & VALIDATION EXPERTE
# =========================================================
def analyse_gait_physique(sig, fs=25):
    mag = np.sqrt(np.sum(np.square(sig), axis=1))
    mag = mag - np.mean(mag)
    corr = np.correlate(mag, mag, mode='full')[len(mag) - 1:]
    corr /= np.max(corr)
    min_l, max_l = int(0.4 * fs), int(1.2 * fs)
    if len(corr) > max_l:
        p_idx = np.argmax(corr[min_l:max_l]) + min_l
        return corr[p_idx], p_idx / fs
    return 0, 1e-7

def full_expert_validation(old_label_id, nom_activite):
    label_cible = label_mapping[old_label_id]
    indices = np.where(y_train == label_cible)[0]
    zm_label, _ = vae_enc.predict(X_train[indices[:200]], verbose=0)
    z_centre = np.mean(zm_label, axis=0)
    z_std    = np.std(zm_label, axis=0)

    # * 1.5 pour couvrir plus de variabilité intra-classe
    z_gen = np.random.normal(loc=z_centre, scale=z_std * 1.5, size=(30, latent_dim))
    gen_windows = vae_dec.predict(z_gen, verbose=0)
    full_sig = gen_windows.reshape(-1, 3)

    score_gait, step_t = analyse_gait_physique(full_sig)
    f, t_spec, Sxx = signal.spectrogram(full_sig[:, 0], fs=25)

    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(4, 2)

    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(full_sig[:, 0], color='orange', lw=1)
    ax1.set_title(f"Génération 1 min : {nom_activite}")

    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(full_sig[250:375, 0], marker='.', color='red', lw=1)
    ax2.set_title("Zoom 5s (Périodicité)")

    ax3 = fig.add_subplot(gs[1, 1])
    ax3.pcolormesh(t_spec, f, 10 * np.log10(Sxx), shading='gouraud', cmap='magma')
    ax3.set_title("Spectrogramme")

    ax4 = fig.add_subplot(gs[2, :])
    ax4.axis('off')
    inv_map = {v: k for k, v in label_mapping.items()}
    txt = "VERDICTS DES JUGES :\n"
    for m in judges:
        p = m.predict(gen_windows, verbose=0)
        c_new = Counter(np.argmax(p, axis=1)).most_common(1)[0][0]
        c_old = inv_map[c_new]
        confiance = np.mean(np.max(p, axis=1)) * 100
        verdict = '✅ CRÉDIBLE' if c_old == old_label_id else '❌ INCOHÉRENT'
        txt += f"- {m.name}: Label {c_old} | Confiance {confiance:.1f}% | {verdict}\n"

    txt += f"\nSCORE GAIT : {score_gait:.2f}/1.0 | Cadence : {60 / step_t:.1f} pas/min"
    ax4.text(0.05, 0.1, txt, fontsize=12, family='monospace',
             bbox=dict(facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()

# Test Final pour la marche (Label original = 1)
full_expert_validation(old_label_id=1, nom_activite="Marche")
